In [3]:
import pandas as pd

h5_file = "profiles_old.h5"  # Adjust path if located in a subfolder, e.g. "profiles/profiles.h5"

with pd.HDFStore(h5_file, mode="r") as store:
    print("Keys stored in HDF5 file:")
    for key in store.keys():
        df = store[key]
        print(f"\nKey: {key}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  Preview:\n{df.head(2)}")

Keys stored in HDF5 file:

Key: /CF_ATR_profiles_2030
  Shape: (8760, 3)
  Columns: ['Pessimistic', 'Most likely', 'Optimistic']
  Preview:
   Pessimistic  Most likely  Optimistic
0          1.0          1.0         1.0
1          1.0          1.0         1.0

Key: /CF_ATR_profiles_2040
  Shape: (8760, 3)
  Columns: ['Pessimistic', 'Most likely', 'Optimistic']
  Preview:
   Pessimistic  Most likely  Optimistic
0          1.0          1.0         1.0
1          1.0          1.0         1.0

Key: /CF_ATR_profiles_2050
  Shape: (8760, 3)
  Columns: ['Pessimistic', 'Most likely', 'Optimistic']
  Preview:
   Pessimistic  Most likely  Optimistic
0          NaN          NaN         NaN
1          NaN          NaN         NaN

Key: /CF_electrolyser_profiles_2030
  Shape: (8760, 3)
  Columns: ['Pessimistic', 'Most likely', 'Optimistic']
  Preview:
   Pessimistic  Most likely  Optimistic
0          1.0          1.0         1.0
1          1.0          1.0         1.0

Key: /CF_electrolyser_profil

In [2]:
import pickle
from pathlib import Path
import pandas as pd

# If the notebook/session is running in the folder containing the file:
file_path = Path.cwd() / "profiles.pkl"

# Or if you know the exact path:
# file_path = Path(r"C:\path\to\your\folder\profiles.pkl")

with open(file_path, "rb") as f:
    profiles = pickle.load(f)

# View keys
print(list(profiles.keys()))

# Inspect a profile
profiles["electricity_price_profiles_2030"].head()

['electricity_price_profiles_2030', 'electricity_price_profiles_2040', 'electricity_price_profiles_2050', 'hydrogen_price_profiles_2030', 'hydrogen_price_profiles_2040', 'hydrogen_price_profiles_2050', 'naturalgas_price_profiles_2030', 'naturalgas_price_profiles_2040', 'naturalgas_price_profiles_2050', 'CF_electrolyser_profiles_2030', 'CF_electrolyser_profiles_2040', 'CF_electrolyser_profiles_2050', 'CF_ATR_profiles_2030', 'CF_ATR_profiles_2040', 'CF_ATR_profiles_2050', 'df_wind_profiles_HUBN', 'df_wind_profiles_HUBE', 'df_wind_profiles_HUBW', 'df_solar_profiles_NED']


,Pessimistic,Most likely,Optimistic
0,29.22294,29.22294,29.222940
1,29.22294,29.22294,26.414940
2,29.22294,29.22294,26.405522
3,29.22294,29.22294,26.405522
4,29.22294,29.22294,26.414940


In [4]:
import pickle
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal

# Adjust paths as needed
base_path = Path.cwd()
pkl_path = base_path / "profiles.pkl"
h5_path = base_path / "profiles_old.h5"

# 1. Load both sources
with open(pkl_path, "rb") as f:
    profiles_pkl = pickle.load(f)

# Read all keys and dataframes from the HDF5 file
profiles_h5 = {}
with pd.HDFStore(h5_path, mode="r") as store:
    for key in store.keys():
        clean_key = key.lstrip("/")  # Strip leading slash to match pickle keys
        profiles_h5[clean_key] = store.get(key)

# 2. Key alignment check
pkl_keys = set(profiles_pkl.keys())
h5_keys = set(profiles_h5.keys())

print(f"Total keys in Pickle: {len(pkl_keys)}")
print(f"Total keys in HDF5:   {len(h5_keys)}")

missing_in_h5 = pkl_keys - h5_keys
missing_in_pkl = h5_keys - pkl_keys

if missing_in_h5:
    print(f"\nKeys only in Pickle: {missing_in_h5}")
if missing_in_pkl:
    print(f"Keys only in HDF5:   {missing_in_pkl}")

# 3. Value-by-value comparison
print("\n" + "=" * 60)
print(f"{'Profile Key':<35} | {'Shape':<12} | Status")
print("=" * 60)

common_keys = sorted(pkl_keys.intersection(h5_keys))

for key in common_keys:
    df_pkl = profiles_pkl[key]
    df_h5 = profiles_h5[key]

    shape_str = f"{df_pkl.shape[0]}x{df_pkl.shape[1]}"

    # Check shape mismatch
    if df_pkl.shape != df_h5.shape:
        print(
            f"{key:<35} | {shape_str:<12} | MISMATCH: Shape differs ({df_h5.shape})"
        )
        continue

    # Numerical & column check
    try:
        assert_frame_equal(
            df_pkl,
            df_h5,
            check_dtype=False,  # Set False if float32 vs float64 differs across formats
            check_exact=False,
            rtol=1e-4,
            atol=1e-4,
        )
        print(f"{key:<35} | {shape_str:<12} | MATCH")
    except AssertionError as e:
        # Calculate maximum absolute difference if columns match
        if list(df_pkl.columns) == list(df_h5.columns):
            max_diff = (df_pkl - df_h5).abs().max().max()
            print(f"{key:<35} | {shape_str:<12} | VALUE DIFF (Max Δ: {max_diff:.4e})")
        else:
            print(f"{key:<35} | {shape_str:<12} | COLUMN NAME MISMATCH")

Total keys in Pickle: 19
Total keys in HDF5:   19

Profile Key                         | Shape        | Status
CF_ATR_profiles_2030                | 8760x3       | MATCH
CF_ATR_profiles_2040                | 8760x3       | MATCH
CF_ATR_profiles_2050                | 8760x3       | MATCH
CF_electrolyser_profiles_2030       | 8760x3       | MATCH
CF_electrolyser_profiles_2040       | 8760x3       | MATCH
CF_electrolyser_profiles_2050       | 8760x3       | MATCH
df_solar_profiles_NED               | 8760x3       | MATCH
df_wind_profiles_HUBE               | 8760x3       | VALUE DIFF (Max Δ: 1.0000e+00)
df_wind_profiles_HUBN               | 8760x3       | MATCH
df_wind_profiles_HUBW               | 8760x3       | MATCH
electricity_price_profiles_2030     | 8760x3       | MATCH
electricity_price_profiles_2040     | 8760x3       | MATCH
electricity_price_profiles_2050     | 8760x3       | MATCH
hydrogen_price_profiles_2030        | 8760x3       | MATCH
hydrogen_price_profiles_2040        | 

In [5]:
# In your old HDF5 file, HUBE and HUBN should be identical if the typo was present
is_old_hube_actually_hubn = (
    profiles_h5["df_wind_profiles_HUBE"].equals(
        profiles_h5["df_wind_profiles_HUBN"]
    )
)
print(f"Was old HUBE identical to old HUBN? {is_old_hube_actually_hubn}")

Was old HUBE identical to old HUBN? True
